# Object Detection Metrics: Bounding Box Geometry, IoU, and Average Precision (AP)

**Session:** 2026-10-31 · Competition tasks and metrics  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

Object detection marries classification with spatial localization: the model must determine **what** objects are present and **where** they reside in pixel space. Because bounding box coordinates are continuous while detections are discrete, standard classification metrics cannot be applied directly. We must first establish spatial overlap using **Intersection over Union (IoU)**, match candidate proposals to ground truth boxes, and compute **Average Precision (AP)** across confidence rankings.

## Learning Outcomes

- Transform bounding boxes between Pascal VOC `[xmin, ymin, xmax, ymax]`, COCO `[xmin, ymin, w, h]`, and YOLO formats.
- Derive and implement Intersection over Union (IoU) from geometric coordinates.
- Implement vectorized batch IoU matrices for comparing $N$ proposals with $M$ targets.
- Match candidate detections to ground truth targets using IoU thresholding (e.g. $\text{IoU} \ge 0.50$).
- Construct Precision-Recall curves sorted by detector confidence.
- Compute Average Precision (AP) using the 11-point interpolated and all-point Area Under Curve methods.


In [ ]:
from collections.abc import Sequence


# 1. Bounding Box Format Conversion Utilities
def voc_to_coco(box: Sequence[float]) -> list[float]:
    """Convert [xmin, ymin, xmax, ymax] to [xmin, ymin, width, height]."""
    xmin, ymin, xmax, ymax = box
    return [xmin, ymin, xmax - xmin, ymax - ymin]


def coco_to_voc(box: Sequence[float]) -> list[float]:
    """Convert [xmin, ymin, width, height] to [xmin, ymin, xmax, ymax]."""
    xmin, ymin, w, h = box
    return [xmin, ymin, xmin + w, ymin + h]


def voc_to_yolo(box: Sequence[float], img_w: float, img_h: float) -> list[float]:
    """Convert [xmin, ymin, xmax, ymax] to normalized [x_center, y_center, w, h]."""
    xmin, ymin, xmax, ymax = box
    xc = (xmin + xmax) / (2.0 * img_w)
    yc = (ymin + ymax) / (2.0 * img_h)
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h
    return [xc, yc, w, h]


sample_box = [50, 60, 200, 250]
print(f"VOC Format:  {sample_box}")
print(f"COCO Format: {voc_to_coco(sample_box)}")
print(f"YOLO Format: {[round(x, 4) for x in voc_to_yolo(sample_box, 640, 480)]}")

## 2. Intersection over Union (IoU) Geometric Derivation

Given two bounding boxes $A = (x_1^A, y_1^A, x_2^A, y_2^A)$ and $B = (x_1^B, y_1^B, x_2^B, y_2^B)$:
1. The intersection coordinates are:
   $$x_1^I = \max(x_1^A, x_1^B), \quad y_1^I = \max(y_1^A, y_1^B)$$
   $$x_2^I = \min(x_2^A, x_2^B), \quad y_2^I = \min(y_2^A, y_2^B)$$
2. The intersection dimensions and area are:
   $$w_I = \max(0, x_2^I - x_1^I), \quad h_I = \max(0, y_2^I - y_1^I)$$
   $$\text{Area}_I = w_I \times h_I$$
3. The union area is:
   $$\text{Area}_U = \text{Area}_A + \text{Area}_B - \text{Area}_I$$
4. $\text{IoU} = \frac{\text{Area}_I}{\text{Area}_U}$


In [ ]:
from collections.abc import Sequence


def calculate_iou(box_a: Sequence[float], box_b: Sequence[float]) -> float:
    """Calculate Intersection over Union (IoU) between two VOC boxes [xmin, ymin, xmax, ymax]."""
    ix_min = max(box_a[0], box_b[0])
    iy_min = max(box_a[1], box_b[1])
    ix_max = min(box_a[2], box_b[2])
    iy_max = min(box_a[3], box_b[3])

    iw = max(0.0, ix_max - ix_min)
    ih = max(0.0, iy_max - iy_min)
    intersection = iw * ih

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - intersection

    if union <= 0.0:
        return 0.0
    return float(intersection / union)


# Test cases
gt = [70, 80, 120, 150]
perfect_match = [70, 80, 120, 150]
partial_match = [50, 50, 90, 100]
no_match = [300, 300, 400, 400]

print(f"IoU (Perfect overlap): {calculate_iou(gt, perfect_match):.4f}")
print(f"IoU (Partial overlap): {calculate_iou(gt, partial_match):.4f}")
print(f"IoU (Disjoint boxes):  {calculate_iou(gt, no_match):.4f}")

## 3. Vectorized Batch IoU Matrix Computation

Detectors generate hundreds of proposals per image. Computing pairwise IoU sequentially is slow ($O(NM)$). We vectorize the computation using NumPy broadcasting.


In [ ]:
import numpy as np


def batch_iou(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """Vectorized pairwise IoU between N boxes (boxes_a) and M boxes (boxes_b).

    boxes_a: shape (N, 4) in [xmin, ymin, xmax, ymax]
    boxes_b: shape (M, 4) in [xmin, ymin, xmax, ymax]
    Returns: shape (N, M) of IoU scores
    """
    tl = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])  # (N, M, 2)
    br = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])  # (N, M, 2)

    wh = np.clip(br - tl, 0.0, None)  # (N, M, 2)
    intersection = wh[:, :, 0] * wh[:, :, 1]  # (N, M)

    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])  # (N,)
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])  # (M,)
    union = area_a[:, None] + area_b[None, :] - intersection  # (N, M)

    return np.where(union > 0, intersection / union, 0.0)


cand_boxes = np.array([[50, 60, 200, 250], [55, 65, 195, 245], [300, 120, 520, 400]])
gt_boxes = np.array([[50, 60, 200, 250], [300, 120, 520, 400]])

iou_mat = batch_iou(cand_boxes, gt_boxes)
print(f"Pairwise IoU Matrix (Shape {iou_mat.shape}):")
print(iou_mat.round(3))

## 4. Candidate Matching and Average Precision (AP)

1. Sort candidate predictions across all images by confidence in descending order.
2. For each prediction, find the ground-truth box with the highest IoU.
3. If $\text{IoU} \ge \alpha$ (e.g. 0.50) and that ground-truth box has not been claimed, mark as **True Positive (TP)**.
4. If $\text{IoU} < \alpha$ or the ground truth has already been matched, mark as **False Positive (FP)**.
5. Compute running Precision and Recall, and compute the Area Under the Precision-Recall curve.


In [ ]:
import numpy as np


def calculate_average_precision(
    precisions: np.ndarray,
    recalls: np.ndarray,
    method: str = "all_points",
) -> float:
    """Compute Average Precision via 11-point interpolation or all-point AUC."""
    if method == "11_points":
        ap = 0.0
        for t in np.linspace(0.0, 1.0, 11):
            valid_p = precisions[recalls >= t]
            p_interp = np.max(valid_p) if valid_p.size > 0 else 0.0
            ap += p_interp / 11.0
        return float(ap)
    else:  # all points
        m_rec = np.concatenate(([0.0], recalls, [1.0]))
        m_pre = np.concatenate(([0.0], precisions, [0.0]))
        for i in range(len(m_pre) - 2, -1, -1):
            m_pre[i] = max(m_pre[i], m_pre[i + 1])
        idx = np.where(m_rec[1:] != m_rec[:-1])[0]
        ap = np.sum((m_rec[idx + 1] - m_rec[idx]) * m_pre[idx + 1])
        return float(ap)


# Synthetic PR trajectory demonstration
test_p = np.array([1.0, 1.0, 0.8, 0.75, 0.6])
test_r = np.array([0.2, 0.4, 0.6, 0.8, 1.0])

print(f"11-Point AP: {calculate_average_precision(test_p, test_r, '11_points'):.4f}")
print(f"All-Point AP: {calculate_average_precision(test_p, test_r, 'all_points'):.4f}")

## 5. End-to-End Evaluation on `data/detection_bounding_boxes.json`


In [ ]:
import json
from pathlib import Path

import numpy as np

json_path = Path("../data/detection_bounding_boxes.json")
with json_path.open("r", encoding="utf-8") as f:
    detection_dataset = json.load(f)

# Evaluate 'dog' class across all images at IoU threshold = 0.50
target_category = "dog"
all_preds = []
total_gt_count = 0

for img in detection_dataset["images"]:
    img_gt = [g for g in img["ground_truth"] if g["category"] == target_category]
    total_gt_count += len(img_gt)
    for p in img["predictions"]:
        if p["category"] == target_category:
            all_preds.append(
                {
                    "image_id": img["image_id"],
                    "box": p["box"],
                    "confidence": p["confidence"],
                }
            )

all_preds.sort(key=lambda x: x["confidence"], reverse=True)

iou_threshold = 0.50
tp = np.zeros(len(all_preds))
fp = np.zeros(len(all_preds))
matched_gt = set()

for i, pred in enumerate(all_preds):
    img = next(item for item in detection_dataset["images"] if item["image_id"] == pred["image_id"])
    gt_boxes = [g for g in img["ground_truth"] if g["category"] == target_category]

    best_iou = 0.0
    best_gt_idx = -1
    for gt_idx, gt in enumerate(gt_boxes):
        score = calculate_iou(pred["box"], gt["box"])
        if score > best_iou:
            best_iou = score
            best_gt_idx = gt_idx

    gt_key = (pred["image_id"], best_gt_idx)
    if best_iou >= iou_threshold and gt_key not in matched_gt:
        tp[i] = 1.0
        matched_gt.add(gt_key)
    else:
        fp[i] = 1.0

cum_tp = np.cumsum(tp)
cum_fp = np.cumsum(fp)
precisions = cum_tp / (cum_tp + cum_fp)
recalls = cum_tp / total_gt_count

ap_score = calculate_average_precision(precisions, recalls, method="all_points")

print(f"Category:                   {target_category}")
print(f"Total Ground Truth Boxes:   {total_gt_count}")
print(f"Total Detections Evaluated: {len(all_preds)}")
print(f"True Positives:             {int(np.sum(tp))}")
print(f"False Positives:            {int(np.sum(fp))}")
print(f"Average Precision (AP@50):  {ap_score:.4f}")